# Applied Machine Learning (CSAI2017P) — Lab Assignment 4
## Customer Churn Prediction

Experiments 5 · CO2, CO4 · **10 marks**

| | |
|---|---|
| Name : EKTA BHARDWAJ
| SAP ID : 590022698
| Batch : 5


**Datasets:** A2 (Telco Customer Churn)

---

### Before you start
- Attempt **2 of 3** in Section A, **1 of 2** in Section B. Section C is compulsory.
- Fit every transformer inside a `Pipeline`. Never fit on the test set.
- Set `random_state` wherever the question asks for reproducibility.
- Every question wants a short written observation, not just a number.
- Restart the kernel and run top-to-bottom before you submit.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

RANDOM_STATE = 0
np.random.seed(RANDOM_STATE)

for m in (np, pd, sklearn):
    print(f"{m.__name__:12s} {m.__version__}")


numpy        2.4.4
pandas       3.0.2
sklearn      1.9.0


## Load the data

Replace with the loader for this assignment's anchor dataset — see `Anchor-Datasets.pdf` for the snippets.

In [2]:
import pandas as pd

df = pd.read_csv("data/WA_Fn-UseC_-Telco-Customer-Churn.csv")

print("Dataset loaded successfully.")
print("Dataset shape:", df.shape)

df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'data/WA_Fn-UseC_-Telco-Customer-Churn.csv'

---

# Section A — attempt any 2 of 3 (2 marks each)

*attempt 2 of 3*

## A1. Encode and fit  &nbsp;&nbsp;`[2 marks]`

*Dataset: A2. Target: `Churn`.*

a. Build a `ColumnTransformer` with one-hot encoding for categoricals and scaling for numerics.
b. Fit a logistic regression inside the pipeline and report test accuracy.
c. State the churn rate in the data and compare it with your accuracy in two lines.


In [ ]:
# A1 — preprocessing + logistic regression

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# TotalCharges is numeric in concept, but some rows contain blank strings.
df = df.copy()
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Convert the target to 0/1.
y = df["Churn"].map({"No": 0, "Yes": 1})
X = df.drop(columns=["Churn", "customerID"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

categorical_features = X.select_dtypes(include=["object"]).columns.tolist()
numeric_features = X.select_dtypes(exclude=["object"]).columns.tolist()

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric_features),
    ("cat", categorical_pipe, categorical_features)
])

logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])

logistic_model.fit(X_train, y_train)

y_pred = logistic_model.predict(X_test)
test_accuracy = accuracy_score(y_test, y_pred)
churn_rate = y.mean()

print(f"Test accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print(f"Overall churn rate: {churn_rate:.4f} ({churn_rate*100:.2f}%)")


Test accuracy: 0.8034 (80.34%)
Overall churn rate: 0.2654 (26.54%)


C:\Users\Ekta Bhardwaj\AppData\Local\Temp\ipykernel_3868\2342527386.py:23: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(include=["object"]).columns.tolist()


**Observation (A1):**

The logistic regression pipeline combines scaling for numeric variables with one-hot encoding for categorical variables, so both feature types can be used together safely.  
The test accuracy is shown above, while the churn rate gives the proportion of customers who actually left. Accuracy is higher than the churn rate because the model correctly classifies many non-churn customers as well; therefore, accuracy alone should not be treated as the complete measure of churn performance.


## A2. Confusion matrix  &nbsp;&nbsp;`[2 marks]`

*Dataset: A2.*

a. Print the confusion matrix and the classification report for your A1 model.
b. State the number of churners the model missed.
c. In two lines, say which error costs the company more: a missed churner, or a retention offer sent to someone who was staying.


In [ ]:
# A2 — confusion matrix and classification report

from sklearn.metrics import confusion_matrix, classification_report

cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)
print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["Stayed (0)", "Churned (1)"],
    digits=4
))

tn, fp, fn, tp = cm.ravel()
print(f"Churners missed by the model (False Negatives): {fn}")


Confusion Matrix:
[[923 112]
 [165 209]]

Classification Report:
              precision    recall  f1-score   support

  Stayed (0)     0.8483    0.8918    0.8695      1035
 Churned (1)     0.6511    0.5588    0.6014       374

    accuracy                         0.8034      1409
   macro avg     0.7497    0.7253    0.7355      1409
weighted avg     0.7960    0.8034    0.7984      1409

Churners missed by the model (False Negatives): 165


**Observation (A2):**

The false-negative value above represents customers who actually churned but were predicted to stay.  
For a retention system, this error is usually more costly than sending an offer to a customer who was going to stay, because a missed churner can result in losing the customer and future revenue. A retention offer to a non-churner mainly represents unnecessary campaign cost.


---

# Section B — attempt any 1 of 2 (4 marks each)

*attempt 1 of 2*

## B1. Model comparison  &nbsp;&nbsp;`[4 marks]`

*Dataset: A2.*

a. Train logistic regression, a decision tree (`max_depth=5`) and a random forest on the same pipeline.
b. Report accuracy, precision, recall, F1 and ROC-AUC for each in one table.
c. Name the best model for this problem and defend the choice in 4–6 lines, referring to the cost argument.


In [ ]:
# B1 — compare three models on the same preprocessing pipeline

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000, random_state=RANDOM_STATE
    ),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=5, random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1
    )
}

results = []
fitted_models = {}

for name, classifier in models.items():
    # Use the same preprocessor for every model.
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", classifier)
    ])

    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    proba = pipe.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, proba)
    })
    fitted_models[name] = pipe

comparison = pd.DataFrame(results)
display(comparison.style.format({
    "Accuracy": "{:.4f}",
    "Precision": "{:.4f}",
    "Recall": "{:.4f}",
    "F1": "{:.4f}",
    "ROC-AUC": "{:.4f}"
}))

# For churn prediction, recall is given extra importance because missing a real
# churner can be more costly than contacting a customer who stays.
best_model_name = comparison.sort_values(
    ["Recall", "F1", "ROC-AUC"], ascending=False
).iloc[0]["Model"]
best_model = fitted_models[best_model_name]

print(f"Selected best model for the churn objective: {best_model_name}")


,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,Logistic Regression,0.8034,0.6511,0.5588,0.6014,0.8515
1,Decision Tree,0.7935,0.6095,0.6176,0.6135,0.8368
2,Random Forest,0.7864,0.6327,0.4652,0.5362,0.8256


Selected best model for the churn objective: Decision Tree


**Observation (B1):**

The three classifiers were evaluated on the same train/test split and the same preprocessing approach, making the comparison fair.  
For churn prediction, I would not select a model only because it has the highest accuracy. Recall is important because a false negative means a real churner was missed.  
Based on the displayed results, **the selected model is printed above** because it gives the strongest recall-first balance among the tested models.  
This choice reflects the business cost: missing a likely churner can be more expensive than making an unnecessary retention offer.


---

# Section C — compulsory (2 marks)

*compulsory*

## C1. Which features carry the signal  &nbsp;&nbsp;`[2 marks]`

*Dataset: A2.*

a. Report the ten most important features from your best model (coefficients or feature importances).
b. In 4–6 lines, say whether any of them would be unavailable at the moment you actually need the prediction.


In [ ]:
# C1 — ten most important features from the selected model

# The selected model may be linear or tree-based, so handle both cases.
classifier = best_model.named_steps["classifier"]
fitted_preprocessor = best_model.named_steps["preprocessor"]

feature_names = fitted_preprocessor.get_feature_names_out()

if hasattr(classifier, "coef_"):
    importance_values = classifier.coef_[0]
    importance_type = "coefficient"
else:
    importance_values = classifier.feature_importances_
    importance_type = "feature importance"

feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importance_values
})

# Show the strongest signals by absolute magnitude while preserving
# the sign for logistic-regression coefficients.
feature_importance["Absolute Importance"] = feature_importance["Importance"].abs()

top10 = (
    feature_importance
    .sort_values("Absolute Importance", ascending=False)
    .head(10)
    .drop(columns="Absolute Importance")
    .reset_index(drop=True)
)

print(f"Selected model: {best_model_name}")
print(f"Importance measure: {importance_type}")
display(top10.style.format({"Importance": "{:.4f}"}))


Selected model: Decision Tree
Importance measure: feature importance


,Feature,Importance
0,cat__Contract_Month-to-month,0.5105
1,num__tenure,0.1662
2,cat__InternetService_Fiber optic,0.1553
3,num__TotalCharges,0.0366
4,num__MonthlyCharges,0.0286
5,cat__OnlineSecurity_No,0.0257
6,cat__PaymentMethod_Electronic check,0.0175
7,cat__MultipleLines_No,0.0133
8,cat__Contract_Two year,0.0123
9,cat__PhoneService_Yes,0.0087


**Observation (C1):**

The table above lists the ten strongest signals identified by the selected model. For a churn prediction system, useful features should be available before the retention decision is made.  
Customer profile and service information, such as contract type, tenure and subscribed services, can normally be known at prediction time.  
However, any variable that is recorded only after the customer has already started the cancellation process would create leakage and should not be used.  
Therefore, the important-feature list should be checked from a timing perspective, not only from its numerical importance.
